In [1]:
import os, sys, logging
logging.basicConfig(stream=sys.stdout, 
                    format='%(asctime)s %(message)s',
                    level=logging.INFO)

In [2]:
from rdflib import URIRef

database_url = 'http://localhost:3030/dbpedia_en'
identifier = URIRef("https://mangosaurus.eu/dbpedia")

In [3]:
from rdflib.plugins.stores.sparqlstore import SPARQLUpdateStore
from nifigator import NifGraph

# Connect to triplestore
store = SPARQLUpdateStore(
    query_endpoint = database_url+'/sparql',
    update_endpoint = database_url+'/update'
)
# Create NifVectorGraph with this store
g = NifGraph(
    store=store, 
    identifier=identifier
)

In [4]:
import pickle

with open('..//data//dbpedia_phrases_10000.pickle', 'rb') as handle:
    v_phrases = pickle.load(handle)
with open('..//data//dbpedia_contexts_10000.pickle', 'rb') as handle:
    v_contexts = pickle.load(handle)
with open('..//data//dbpedia_lemmas_10000.pickle', 'rb') as handle:
    lemmas_dict = pickle.load(handle)

In [5]:
lang = "en"

stop_words_en = [
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of',
    'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off',
    'over', 'under', 'further',
    'per',
]

stop_words_nl = [
    'een', 'de', 'het', 'en', 'maar', 'als', 'of', 'omdat', 'van',
    'te', 'in', 'op', 'aan', 'met', 'voor', 'er', 'om', 'dan', 'of',
    'door', 'over', 'bij', 'ook', 'tot', 'uit', 'naar', 'want', 'nog',
    'toch', 'al', 'dus', 'onder', 'tegen', 'na', 'reeds',
]
# set the parameters to create the NifVector graph
params = {
    "words_filter": {
        "data": stop_words_en if lang=="en" else stop_words_nl,
        "name": "nifvec.stopwords"
    },
}

In [6]:
from selmr import SELMR, LanguageMultisets

# construct a SELMR data structure with the DBpedia phrases and contexts
selmr = SELMR(
    multisets=LanguageMultisets(v_phrases, v_contexts, lemmas_dict),
    params=params
)

In [7]:
# Read two documents in DBpedia about Aldebaran and Antares stars
doc1 = g.get(
    URIRef("http://dbpedia.org/resource/Aldebaran?dbpv=2020-07&nif=context")
)
doc2 = g.get(
    URIRef("http://dbpedia.org/resource/Antares?dbpv=2020-07&nif=context")
)

In [155]:
doc1.sentences[1].anchorOf

"It is the brightest star in Taurus and generally the fourteenth-brightest star in the night sky, though it varies slowly in brightness between magnitude 0.75 and 0.95. Aldebaran is believed to host a planet several times the mass of Jupiter, named Aldebaran b. Aldebaran is a red giant, cooler than the sun with a surface temperature of 3,900 K, but its radius is about 44 times the sun's, so it is over 400 times as luminous."

In [223]:
from selmr import process_documents, Multiset

correct = 0
total = 0
for sent in doc1.sentences:
    contexts = process_documents([sent.anchorOf]).contexts
    for item, value in contexts.items():
        m = selmr.phrases(item)
        if len(m) > 0:
            # take highest item in the Counter
            if item[0]+" "+list(m.items())[0][0]+" "+item[1] in sent.anchorOf:
                correct += 1
    total += len(contexts.keys())
print("Correct: "+str(correct / total))

Correct: 0.055794275137171945


In [150]:
for item in selmr.phrases(('of the', 'SENTEND'), topn=None).keys():
    if item.lower()!=item and "King" in item:
        print(item)

United Kingdom
King
Kingdom
Three Kingdoms
Kings of Norway
Kings
Kingdom of Norway
Kingdom of Naples
King of France
Seven Kingdoms
Kingdom of Hungary
Kingdom of Jerusalem
King James Bible
East Frankish Kingdom
Kingdom of Prussia
Northern Kingdom
Kingdom of God
New Kingdom
Kingdom of the Netherlands
Kingdom of Heaven
Bosporan Kingdom had 1 childi
Kingdom of Sicily
Kingdom of Thailand
Zulu Kingdom
Kingdom of Italy
Kings of Alba
Kingdom of Norway in 1035
Kingdom of Galicia
Five Kings
King daughter Elizabeth
Parliament of the United Kingdom
Kingdom of Judah
Kingdom of Hawaii
Crimson King
Kingdom of Yugoslavia
Habsburg Kingdom of Hungary
King ;
Kingdom of Tambapanni
King of England
Kingdom of Tonga
Polish Kingdom
Kingdom of Croatia
United Kingdom in 1761
King mistress Madame de Pompadour
King Bench by royal prerogative
Kingdom of Bohemia
Visigothic Kingdom
Kingdom of Aragon
